In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import os

print("✅ Libraries imported!")

✅ Libraries imported!


In [22]:
df = pd.read_csv('../data/dataset.csv')
# Rename 'Disease' to 'prognosis' 
df = df.rename(columns={'Disease': 'prognosis'})

print("✅ Dataset loaded and column renamed!")
print(f"Shape: {df.shape}")
df.head()

✅ Dataset loaded and column renamed!
Shape: (4920, 18)


,prognosis,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,nodal_skin_eruptions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
#Check for Missing Values
print("=== Missing Values Per Column ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found!")

print(f"\n=== Data Types ===")
print(df.dtypes.value_counts())

=== Missing Values Per Column ===
Symptom_4      348
Symptom_5     1206
Symptom_6     1986
Symptom_7     2652
Symptom_8     2976
Symptom_9     3228
Symptom_10    3408
Symptom_11    3726
Symptom_12    4176
Symptom_13    4416
Symptom_14    4614
Symptom_15    4680
Symptom_16    4728
Symptom_17    4848
dtype: int64

=== Data Types ===
object    18
Name: count, dtype: int64


In [24]:
#Clean the Data
# Strip whitespace from all string columns
df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

# Fill NaN values in symptom columns with 0
symptom_cols = [col for col in df.columns if col != 'prognosis']
df[symptom_cols] = df[symptom_cols].fillna(0)

# Remove duplicate 
before = df.shape[0]
#df = df.drop_duplicates()
after = df.shape[0]

#print(f"✅ Duplicates removed: {before - after} rows dropped")
print(f"✅ Clean dataset shape: {df.shape}")

✅ Clean dataset shape: (4920, 18)


In [25]:
#Encode Symptoms to Numbers
# Most symptom columns are already binary (0/1)
# But let's make sure ALL values are numeric
symptom_cols = [col for col in df.columns if col != 'prognosis']

for col in symptom_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print("✅ All symptom columns converted to numeric (0 or 1)")
print(f"\nSample symptom columns: {symptom_cols[:5]}")
print(f"\nUnique values in symptom cols: {df[symptom_cols[0]].unique()}")

✅ All symptom columns converted to numeric (0 or 1)

Sample symptom columns: ['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5']

Unique values in symptom cols: [0]


In [26]:
#Encode Disease Labels
# ML models need numbers, not text — so we encode disease names
le = LabelEncoder()
df['prognosis_encoded'] = le.fit_transform(df['prognosis'])

print("✅ Disease labels encoded!")
print(f"\nTotal diseases: {len(le.classes_)}")
print("\nEncoding map (first 10):")
for i, disease in enumerate(le.classes_[:10]):
    print(f"  {i} → {disease}")

✅ Disease labels encoded!

Total diseases: 41

Encoding map (first 10):
  0 → (vertigo) Paroymsal  Positional Vertigo
  1 → AIDS
  2 → Acne
  3 → Alcoholic hepatitis
  4 → Allergy
  5 → Arthritis
  6 → Bronchial Asthma
  7 → Cervical spondylosis
  8 → Chicken pox
  9 → Chronic cholestasis


In [27]:
#Split Features and Target
# X = symptoms (input), y = disease (output)
X = df[symptom_cols]
y = df['prognosis_encoded']

print(f"✅ Features (X) shape: {X.shape}")
print(f"✅ Target  (y) shape: {y.shape}")
print(f"\nFeature columns ({len(symptom_cols)} symptoms):")
print(symptom_cols)

✅ Features (X) shape: (4920, 17)
✅ Target  (y) shape: (4920,)

Feature columns (17 symptoms):
['Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']


In [28]:
#Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% training, 20% testing
    random_state=42,     # Makes results reproducible
    stratify=y           # Ensures all diseases appear in both splits
)

print("✅ Data split complete!")
print(f"\nTraining set:  {X_train.shape[0]} samples")
print(f"Testing set:   {X_test.shape[0]} samples")
print(f"Features used: {X_train.shape[1]} symptoms")

✅ Data split complete!

Training set:  3936 samples
Testing set:   984 samples
Features used: 17 symptoms


In [29]:
#Save 
# Create models folder if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save processed data
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save the label encoder 
with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save symptom column list
with open('../models/symptom_columns.pkl', 'wb') as f:
    pickle.dump(symptom_cols, f)

print("✅ All files saved!")
print("\nSaved files:")
print("  📄 data/X_train.csv")
print("  📄 data/X_test.csv")
print("  📄 data/y_train.csv")
print("  📄 data/y_test.csv")
print("  🔧 models/label_encoder.pkl")
print("  🔧 models/symptom_columns.pkl")

✅ All files saved!

Saved files:
  📄 data/X_train.csv
  📄 data/X_test.csv
  📄 data/y_train.csv
  📄 data/y_test.csv
  🔧 models/label_encoder.pkl
  🔧 models/symptom_columns.pkl


In [30]:
#Final Summary
print("=" * 45)
print("      PREPROCESSING COMPLETE ✅")
print("=" * 45)
print(f"  Total samples    : {df.shape[0]}")
print(f"  Symptom features : {len(symptom_cols)}")
print(f"  Diseases         : {len(le.classes_)}")
print(f"  Training samples : {X_train.shape[0]}")
print(f"  Testing samples  : {X_test.shape[0]}")
print("=" * 45)
print("\n🚀 Ready for model training!")

      PREPROCESSING COMPLETE ✅
  Total samples    : 4920
  Symptom features : 17
  Diseases         : 41
  Training samples : 3936
  Testing samples  : 984

🚀 Ready for model training!


SyntaxError: unexpected character after line continuation character (600713393.py, line 1)